# C00 — Narrative → Race Classification v2
**Task**: Given a police narrative, predict the race of the subject (Black / White).

**Models:**
| Model | Type | Purpose |
|---|---|---|
| `microsoft/deberta-v3-small` | open | SOTA small model |
| `mistralai/Mistral-7B-v0.1` | open | Large model |

**Experimental conditions:**
1. Original narratives (upper bound)

**Evaluation**: Balanced accuracy, macro-F1, per-class recall — all with 3-seed average + bootstrap CI

In [1]:

import os
# Force a single visible GPU to prevent Trainer from wrapping with torch.nn.DataParallel.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import sys
sys.path.insert(0, '../../notebooks')
sys.path.insert(0, '../../notebooks/classification')
import config

import json, re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                              classification_report, make_scorer)
from sklearn.calibration import CalibratedClassifierCV
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEEDS = [42, 123, 999]  # 3 seeds for stability
CV_FOLDS = 5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Visible GPUs: {os.environ.get("CUDA_VISIBLE_DEVICES")}')
print(f'torch.cuda.device_count(): {torch.cuda.device_count()}')
print(f'Seeds: {SEEDS}')

# ── One-time DeBERTa-v3 tokenizer fix (Transformers ≥5.x) ─────────────────────
# Transformers 5.x has a bug in convert_to_native_format for DebertaV2Tokenizer:
# it tries to parse the SPM vocab as a tiktoken BPE file.  The fix:
#   1. Generate tokenizer.json from spm.model via DebertaV2Converter (once).
#   2. Remove the .no_exist/tokenizer.json cache marker so HuggingFace Hub uses
#      the generated file instead of skipping it.
import glob as _glob
_hub_root = os.path.expanduser('~/.cache/huggingface/hub')
_deberta_snaps = _glob.glob(
    os.path.join(_hub_root, 'models--microsoft--deberta-v3-small', 'snapshots', '*', 'spm.model')
)
for _spm_path in _deberta_snaps:
    _snap_dir   = os.path.dirname(_spm_path)
    _commit_id  = os.path.basename(_snap_dir)
    _tok_json   = os.path.join(_snap_dir, 'tokenizer.json')
    _no_exist   = os.path.join(_hub_root, 'models--microsoft--deberta-v3-small',
                               '.no_exist', _commit_id, 'tokenizer.json')
    # Remove stale "not found" marker so the generated tokenizer.json is picked up
    if os.path.exists(_no_exist):
        os.remove(_no_exist)
    # Generate tokenizer.json if absent
    if not os.path.exists(_tok_json):
        print(f'Generating DeBERTa tokenizer.json in {_snap_dir} ...')
        from transformers import DebertaV2Tokenizer
        from transformers.convert_slow_tokenizer import DebertaV2Converter
        _slow_tok  = DebertaV2Tokenizer(vocab_file=_spm_path, do_lower_case=False)
        DebertaV2Converter(_slow_tok).converted().save(_tok_json)
        print('  Done.')
del _glob, _hub_root, _deberta_snaps
print('DeBERTa tokenizer cache: OK')


Device: cuda
Visible GPUs: 0
torch.cuda.device_count(): 1
Seeds: [42, 123, 999]
DeBERTa tokenizer cache: OK


## 1. Load Data — Three Conditions

In [2]:
NAR_CANDIDATES = ['narrative_clean', 'narrative_redacted', 'narrative', 'text']

def detect_nar_col(df):
    for c in NAR_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f'No narrative column found. Columns: {list(df.columns)}')

def load_bw(path, nar_col=None):
    with open(path) as f:
        df = pd.DataFrame(json.load(f))
    df = df[df['race'].isin(['Black','White'])].copy()
    df['label'] = (df['race'] == 'Black').astype(int)
    if nar_col is None or nar_col not in df.columns:
        nar_col = detect_nar_col(df)
    df['text'] = df[nar_col].fillna('')
    return df, nar_col

# Condition 1: original narratives
if os.path.exists(config.CLEAN_DATA):
    orig_df, orig_nar_col = load_bw(config.CLEAN_DATA)
elif os.path.exists(config.BW_DATASET):
    orig_df, orig_nar_col = load_bw(config.BW_DATASET)
else:
    orig_df, orig_nar_col = load_bw(config.PRIMARY_DATA)
print(f'Original: {len(orig_df)}  Black={orig_df.label.sum()}  White={(orig_df.label==0).sum()}')
print(f'Narrative column: {orig_nar_col}')

# Condition 2: redacted narratives
red_df = None
if os.path.exists(config.REDACTED_DATA):
    red_df, _ = load_bw(config.REDACTED_DATA)
    print(f'Redacted: {len(red_df)}')
else:
    print('No redacted data — run 00_data_pipeline_v2.ipynb first.')

# Balanced dataset approach:
# 1. Under-sample to balance the dataset
# 2. Train-test split (80/20)

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# balance white vs black before splitting
black_df = orig_df[orig_df['label'] == 1]
white_df = orig_df[orig_df['label'] == 0]
min_class_size = min(len(black_df), len(white_df))

black_downsampled = resample(black_df, replace=False, n_samples=min_class_size, random_state=42)
white_downsampled = resample(white_df, replace=False, n_samples=min_class_size, random_state=42)
balanced_orig_df = pd.concat([black_downsampled, white_downsampled])

train_df, test_df = train_test_split(
    balanced_orig_df, test_size=0.2, stratify=balanced_orig_df['label'], random_state=42
)

print(f'\nTotal balanced: {len(balanced_orig_df)}, Train: {len(train_df)} (80%), Test: {len(test_df)} (20%)')
print(f'Train labels: Black={train_df.label.sum()}  White={(train_df.label==0).sum()}')
print(f'Test labels: Black={test_df.label.sum()}  White={(test_df.label==0).sum()}')

# Verify zero CR overlap
if 'cr_number' in train_df.columns:
    overlap = set(train_df.cr_number) & set(test_df.cr_number)
    print(f'CR overlap: {len(overlap)} (must be 0)')

Original: 2656  Black=1952  White=704
Narrative column: narrative
Redacted: 1000

Total balanced: 1408, Train: 1126 (80%), Test: 282 (20%)
Train labels: Black=563  White=563
Test labels: Black=141  White=141
CR overlap: 0 (must be 0)


## 3. Neural Models — Multi-Seed Fine-Tuning

In [3]:

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from transformers.utils.notebook import NotebookProgressCallback
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from torch.utils.data import Dataset

class NarrativeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(
            list(texts), truncation=True, max_length=max_len,
            padding='max_length', return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels': self.labels[i]}
        # Include token_type_ids if the tokenizer produced them
        if 'token_type_ids' in self.enc:
            item['token_type_ids'] = self.enc['token_type_ids'][i]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

def train_classifier(model_name, train_texts, train_labels, test_texts, test_labels,
                      seed=42, output_dir=None):
    """Fine-tune a transformer classifier. Returns balanced accuracy on test set."""
    if output_dir is None:
        output_dir = os.path.join(config.BERT_DIR, f'{model_name.split("/")[-1]}_seed{seed}')

    is_large = any(x in model_name.lower() for x in ['mistral', 'llama', '7b', '13b'])
    has_gpu  = torch.cuda.is_available()
    supports_bf16 = has_gpu and torch.cuda.is_bf16_supported()

    # Transformers ≥5.x routes DeBERTa-v3's SPM tokenizer through a broken
    # tiktoken conversion path. Using the model name bypasses the issue because
    # the generated tokenizer.json in the local cache is picked up via the
    # glob resolution and passed as an absolute path to convert_to_native_format.
    # (No use_fast flag needed — the resolved absolute path is what matters.)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # Only set pad_token for decoder-only models (Mistral/LLaMA) that lack one.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if is_large:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2,
            id2label={0:'White',1:'Black'}, label2id={'White':0,'Black':1},
            quantization_config=bnb_cfg,
            device_map={"": 0},
        )
        model.config.pad_token_id = model.config.eos_token_id
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        peft_cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"], bias="none",
        )
        model = get_peft_model(model, peft_cfg)
        model.is_parallelizable = True
        model.model_parallel = True
        model.print_trainable_parameters()
    else:
        # Force float32: DeBERTa-v3-small weights are stored in float16 on the
        # Hub. Without an explicit torch_dtype, the model loads in float16 and
        # trains without a GradScaler (fp16=False), causing gradient underflow
        # (silent NaN/zero loss). float32 gives numerically stable training.
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2,
            id2label={0:'White',1:'Black'}, label2id={'White':0,'Black':1},
            ignore_mismatched_sizes=True,
            torch_dtype=torch.float32,
        )

    # DeBERTa-v3 disentangled attention overflows at 512 tokens — cap at 256.
    max_len = 256
    train_ds = NarrativeDataset(train_texts, train_labels, tokenizer, max_len=max_len)
    test_ds  = NarrativeDataset(test_texts,  test_labels,  tokenizer, max_len=max_len)

    bsz = 1 if is_large else 16
    num_epochs = 3 if is_large else 5
    total_steps = (len(train_ds) // bsz) * num_epochs
    warmup_steps = max(1, int(0.1 * total_steps))

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=bsz,
        per_device_eval_batch_size=max(2, bsz * 2),
        gradient_accumulation_steps=16 if is_large else 1,
        # DeBERTa-v3-small: 3e-5 is the standard recommended LR.
        learning_rate=2e-4 if is_large else 3e-5,
        warmup_steps=warmup_steps, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch',
        logging_strategy='epoch', logging_first_step=False,
        load_best_model_at_end=True, metric_for_best_model='balanced_accuracy',
        greater_is_better=True,
        # fp16=False: model is loaded in float32 (torch_dtype=torch.float32).
        # DeBERTa-v3 also doesn't work with FP16 AMP due to its disentangled
        # attention. BF16 is used for large models (Mistral) when supported.
        fp16=False,
        bf16=is_large and supports_bf16,
        gradient_checkpointing=is_large,
        max_grad_norm=1.0,
        optim='paged_adamw_8bit' if is_large else 'adamw_torch',
        seed=seed, report_to='none',
        disable_tqdm=True,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=test_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] if not is_large else None,
    )
    # Remove NotebookProgressCallback — per-step widget updates cause ZMQ flush timeout.
    trainer.remove_callback(NotebookProgressCallback)

    trainer.train()
    pred_out = trainer.predict(test_ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    ba = balanced_accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds, average='macro')
    if not (np.isfinite(ba) and np.isfinite(f1)):
        raise RuntimeError(f'Non-finite metric: {model_name} seed={seed}: BA={ba}, F1={f1}')
    report = classification_report(test_labels, preds, target_names=['White','Black'], output_dict=True)
    print(classification_report(test_labels, preds, target_names=['White','Black']))
    white_recall = report['White']['recall']
    black_recall = report['Black']['recall']
    return ba, f1, white_recall, black_recall, trainer

print('Training setup ready.')


Training setup ready.


In [4]:

# DeBERTa-only training cell (run this first)
import gc

train_texts  = train_df['text'].tolist()
test_texts   = test_df['text'].tolist()
train_labels = train_df['label'].tolist()
test_labels  = test_df['label'].tolist()

# Shared container used by the next Mistral cell too
if 'neural_results' not in globals() or not isinstance(neural_results, dict):
    neural_results = {}

model_key = 'deberta-v3-small'
model_name = 'microsoft/deberta-v3-small'

print(f'\n{"="*60}')
print(f'Training: {model_name}')
seed_ba, seed_f1, seed_white_recall, seed_black_recall = [], [], [], []
failures = []

for seed in SEEDS:
    print(f'  Seed {seed}...')
    try:
        ba, f1, white_r, black_r, _ = train_classifier(
            model_name, train_texts, train_labels,
            test_texts, test_labels, seed=seed
        )
        if np.isfinite(ba) and np.isfinite(f1):
            seed_ba.append(float(ba))
            seed_f1.append(float(f1))
            seed_white_recall.append(float(white_r))
            seed_black_recall.append(float(black_r))
            print(f'  Seed {seed}: BA={ba:.4f}, F1={f1:.4f}, White-R={white_r:.3f}, Black-R={black_r:.3f}')
        else:
            failures.append({'seed': seed, 'error': f'Non-finite metrics BA={ba}, F1={f1}'})
            print(f'  Seed {seed} failed: non-finite metrics')
    except KeyboardInterrupt:
        raise
    except Exception as e:
        failures.append({'seed': seed, 'error': str(e)})
        print(f'  Seed {seed} failed: {e}')
    finally:
        # Release GPU memory after each seed so the next seed / cell starts clean.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if len(seed_ba) > 0:
    mean_ba, std_ba = float(np.mean(seed_ba)), float(np.std(seed_ba))
    mean_f1, std_f1 = float(np.mean(seed_f1)), float(np.std(seed_f1))
    print(f'\n  {model_key}: BA={mean_ba:.4f} ± {std_ba:.4f}')
else:
    mean_ba = std_ba = mean_f1 = std_f1 = None
    print(f'\n  {model_key}: no successful seeds (all failed)')

neural_results[model_key] = {
    'mean_ba': mean_ba, 'std_ba': std_ba,
    'mean_f1': mean_f1, 'std_f1': std_f1,
    'seed_ba': seed_ba, 'seed_f1': seed_f1,
    'mean_white_recall': float(np.mean(seed_white_recall)) if seed_white_recall else None,
    'mean_black_recall': float(np.mean(seed_black_recall)) if seed_black_recall else None,
    'seed_white_recall': seed_white_recall,
    'seed_black_recall': seed_black_recall,
    'failures': failures,
    'model_name': model_name,
}

# Final cleanup before Mistral cell
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\nDeBERTa done. GPU cache cleared. Run the next cell for Mistral.')



Training: microsoft/deberta-v3-small
  Seed 42...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

{'loss': '0.6997', 'grad_norm': '5.478', 'learning_rate': '2.672e-05', 'epoch': '1'}
{'eval_loss': '0.6908', 'eval_balanced_accuracy': '0.5532', 'eval_f1_macro': '0.5505', 'eval_runtime': '0.7032', 'eval_samples_per_second': '401', 'eval_steps_per_second': '12.8', 'epoch': '1'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.4966', 'grad_norm': '10.09', 'learning_rate': '9.375e-08', 'epoch': '5'}
{'eval_loss': '0.7639', 'eval_balanced_accuracy': '0.5603', 'eval_f1_macro': '0.5538', 'eval_runtime': '0.7242', 'eval_samples_per_second': '389.4', 'eval_steps_per_second': '12.43', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

{'train_runtime': '71.1', 'train_samples_per_second': '79.18', 'train_steps_per_second': '4.993', 'train_loss': '0.6293', 'epoch': '5'}
              precision    recall  f1-score   support

       White       0.58      0.44      0.50       141
       Black       0.55      0.68      0.61       141

    accuracy                           0.56       282
   macro avg       0.56      0.56      0.55       282
weighted avg       0.56      0.56      0.55       282

  Seed 999: BA=0.5603, F1=0.5538

  deberta-v3-small: BA=0.5816 ± 0.0153

DeBERTa done. GPU cache cleared. Run the next cell for Mistral.


In [5]:

# Mistral-only training cell (run after DeBERTa cell)
import gc

# Free any lingering GPU memory from DeBERTa before loading Mistral-7B.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f'GPU 0 free before Mistral load: {free_bytes/1e9:.1f} GB / {total_bytes/1e9:.1f} GB total')

if 'train_df' not in globals() or 'test_df' not in globals():
    raise RuntimeError('Run data-loading/split cells first.')

train_texts  = train_df['text'].tolist()
test_texts   = test_df['text'].tolist()
train_labels = train_df['label'].tolist()
test_labels  = test_df['label'].tolist()

if 'neural_results' not in globals() or not isinstance(neural_results, dict):
    neural_results = {}

model_key = 'mistral-7b'
model_name = 'mistralai/Mistral-7B-v0.1'

print(f'\n{"="*60}')
print(f'Training: {model_name}')
seed_ba, seed_f1, seed_white_recall, seed_black_recall = [], [], [], []
failures = []

for seed in SEEDS:
    print(f'  Seed {seed}...')
    try:
        ba, f1, white_r, black_r, _ = train_classifier(
            model_name, train_texts, train_labels,
            test_texts, test_labels, seed=seed
        )
        if np.isfinite(ba) and np.isfinite(f1):
            seed_ba.append(float(ba))
            seed_f1.append(float(f1))
            seed_white_recall.append(float(white_r))
            seed_black_recall.append(float(black_r))
            print(f'  Seed {seed}: BA={ba:.4f}, F1={f1:.4f}, White-R={white_r:.3f}, Black-R={black_r:.3f}')
        else:
            failures.append({'seed': seed, 'error': f'Non-finite metrics BA={ba}, F1={f1}'})
            print(f'  Seed {seed} failed: non-finite metrics')
    except KeyboardInterrupt:
        raise
    except Exception as e:
        failures.append({'seed': seed, 'error': str(e)})
        print(f'  Seed {seed} failed: {e}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if len(seed_ba) > 0:
    mean_ba, std_ba = float(np.mean(seed_ba)), float(np.std(seed_ba))
    mean_f1, std_f1 = float(np.mean(seed_f1)), float(np.std(seed_f1))
    print(f'\n  {model_key}: BA={mean_ba:.4f} ± {std_ba:.4f}')
else:
    mean_ba = std_ba = mean_f1 = std_f1 = None
    print(f'\n  {model_key}: no successful seeds (all failed)')

neural_results[model_key] = {
    'mean_ba': mean_ba, 'std_ba': std_ba,
    'mean_f1': mean_f1, 'std_f1': std_f1,
    'seed_ba': seed_ba, 'seed_f1': seed_f1,
    'mean_white_recall': float(np.mean(seed_white_recall)) if seed_white_recall else None,
    'mean_black_recall': float(np.mean(seed_black_recall)) if seed_black_recall else None,
    'seed_white_recall': seed_white_recall,
    'seed_black_recall': seed_black_recall,
    'failures': failures,
    'model_name': model_name,
}

with open(os.path.join(config.CLASS_OUT, 'classification_results_deberta_mistral.json'), 'w') as f:
    json.dump({'neural': neural_results}, f, indent=2)
print('\nMistral done. Results saved.')


GPU 0 free before Mistral load: 45.5 GB / 47.8 GB total

Training: mistralai/Mistral-7B-v0.1
  Seed 42...


use_kernel_func_from_hub is not available in the installed kernels version. Please upgrade kernels to use this feature.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 6,823,936 || all params: 7,117,492,224 || trainable%: 0.0959


...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 6,823,936 || all params: 7,117,492,224 || trainable%: 0.0959


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': '60.03', 'grad_norm': '77.96', 'learning_rate': '4.154e-05', 'epoch': '1'}
{'eval_loss': '2.94', 'eval_balanced_accuracy': '0.4929', 'eval_f1_macro': '0.4911', 'eval_runtime': '35.71', 'eval_samples_per_second': '7.897', 'eval_steps_per_second': '3.949', 'epoch': '1'}
{'loss': '30.8', 'grad_norm': '86.62', 'learning_rate': '8.368e-05', 'epoch': '2'}
{'eval_loss': '2.311', 'eval_balanced_accuracy': '0.5355', 'eval_f1_macro': '0.4914', 'eval_runtime': '35.7', 'eval_samples_per_second': '7.899', 'eval_steps_per_second': '3.95', 'epoch': '2'}
{'loss': '14.32', 'grad_norm': '57.12', 'learning_rate': '0.0001258', 'epoch': '3'}
{'eval_loss': '2.092', 'eval_balanced_accuracy': '0.5213', 'eval_f1_macro': '0.5099', 'eval_runtime': '35.69', 'eval_samples_per_second': '7.901', 'eval_steps_per_second': '3.951', 'epoch': '3'}
{'train_runtime': '1728', 'train_samples_per_second': '1.954', 'train_steps_per_second': '0.123', 'train_loss': '35.05', 'epoch': '3'}
              precision    recal

In [ ]:
# BERT-base-uncased training cell
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if 'train_df' not in globals() or 'test_df' not in globals():
    raise RuntimeError('Run data-loading/split cells first.')

train_texts  = train_df['text'].tolist()
test_texts   = test_df['text'].tolist()
train_labels = train_df['label'].tolist()
test_labels  = test_df['label'].tolist()

if 'neural_results' not in globals() or not isinstance(neural_results, dict):
    neural_results = {}

model_key  = 'bert-base-uncased'
model_name = 'bert-base-uncased'

print(f'\n{"="*60}')
print(f'Training: {model_name}')
seed_ba, seed_f1, seed_white_recall, seed_black_recall = [], [], [], []
failures = []

for seed in SEEDS:
    print(f'  Seed {seed}...')
    try:
        ba, f1, white_r, black_r, _ = train_classifier(
            model_name, train_texts, train_labels,
            test_texts, test_labels, seed=seed
        )
        if np.isfinite(ba) and np.isfinite(f1):
            seed_ba.append(float(ba))
            seed_f1.append(float(f1))
            seed_white_recall.append(float(white_r))
            seed_black_recall.append(float(black_r))
            print(f'  Seed {seed}: BA={ba:.4f}, F1={f1:.4f}, White-R={white_r:.3f}, Black-R={black_r:.3f}')
        else:
            failures.append({'seed': seed, 'error': f'Non-finite metrics BA={ba}, F1={f1}'})
            print(f'  Seed {seed} failed: non-finite metrics')
    except KeyboardInterrupt:
        raise
    except Exception as e:
        failures.append({'seed': seed, 'error': str(e)})
        print(f'  Seed {seed} failed: {e}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if len(seed_ba) > 0:
    mean_ba  = float(np.mean(seed_ba));  std_ba  = float(np.std(seed_ba))
    mean_f1  = float(np.mean(seed_f1));  std_f1  = float(np.std(seed_f1))
    mean_white_r = float(np.mean(seed_white_recall))
    mean_black_r = float(np.mean(seed_black_recall))
    print(f'\n  {model_key}: BA={mean_ba:.4f} ± {std_ba:.4f} | F1={mean_f1:.4f} ± {std_f1:.4f}')
    print(f'  {model_key}: White-R={mean_white_r:.3f} | Black-R={mean_black_r:.3f}')
else:
    mean_ba = std_ba = mean_f1 = std_f1 = mean_white_r = mean_black_r = None
    print(f'\n  {model_key}: no successful seeds')

neural_results[model_key] = {
    'mean_ba': mean_ba, 'std_ba': std_ba,
    'mean_f1': mean_f1, 'std_f1': std_f1,
    'seed_ba': seed_ba, 'seed_f1': seed_f1,
    'mean_white_recall': mean_white_r,
    'mean_black_recall': mean_black_r,
    'seed_white_recall': seed_white_recall,
    'seed_black_recall': seed_black_recall,
    'failures': failures,
    'model_name': model_name,
}

with open(os.path.join(config.CLASS_OUT, 'classification_results_all_models.json'), 'w') as f:
    json.dump({'neural': neural_results}, f, indent=2)
print('\nBERT-base done. Results saved.')


In [ ]:
# LLaMA-3-8B training cell (run after BERT cell)
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f'GPU 0 free before LLaMA load: {free_bytes/1e9:.1f} GB / {total_bytes/1e9:.1f} GB total')

if 'train_df' not in globals() or 'test_df' not in globals():
    raise RuntimeError('Run data-loading/split cells first.')

train_texts  = train_df['text'].tolist()
test_texts   = test_df['text'].tolist()
train_labels = train_df['label'].tolist()
test_labels  = test_df['label'].tolist()

if 'neural_results' not in globals() or not isinstance(neural_results, dict):
    neural_results = {}

model_key  = 'llama3-8b'
model_name = 'meta-llama/Meta-Llama-3-8B'

print(f'\n{"="*60}')
print(f'Training: {model_name}')
seed_ba, seed_f1, seed_white_recall, seed_black_recall = [], [], [], []
failures = []

for seed in SEEDS:
    print(f'  Seed {seed}...')
    try:
        ba, f1, white_r, black_r, _ = train_classifier(
            model_name, train_texts, train_labels,
            test_texts, test_labels, seed=seed
        )
        if np.isfinite(ba) and np.isfinite(f1):
            seed_ba.append(float(ba))
            seed_f1.append(float(f1))
            seed_white_recall.append(float(white_r))
            seed_black_recall.append(float(black_r))
            print(f'  Seed {seed}: BA={ba:.4f}, F1={f1:.4f}, White-R={white_r:.3f}, Black-R={black_r:.3f}')
        else:
            failures.append({'seed': seed, 'error': f'Non-finite metrics BA={ba}, F1={f1}'})
            print(f'  Seed {seed} failed: non-finite metrics')
    except KeyboardInterrupt:
        raise
    except Exception as e:
        failures.append({'seed': seed, 'error': str(e)})
        print(f'  Seed {seed} failed: {e}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if len(seed_ba) > 0:
    mean_ba  = float(np.mean(seed_ba));  std_ba  = float(np.std(seed_ba))
    mean_f1  = float(np.mean(seed_f1));  std_f1  = float(np.std(seed_f1))
    mean_white_r = float(np.mean(seed_white_recall))
    mean_black_r = float(np.mean(seed_black_recall))
    print(f'\n  {model_key}: BA={mean_ba:.4f} ± {std_ba:.4f} | F1={mean_f1:.4f} ± {std_f1:.4f}')
    print(f'  {model_key}: White-R={mean_white_r:.3f} | Black-R={mean_black_r:.3f}')
else:
    mean_ba = std_ba = mean_f1 = std_f1 = mean_white_r = mean_black_r = None
    print(f'\n  {model_key}: no successful seeds')

neural_results[model_key] = {
    'mean_ba': mean_ba, 'std_ba': std_ba,
    'mean_f1': mean_f1, 'std_f1': std_f1,
    'seed_ba': seed_ba, 'seed_f1': seed_f1,
    'mean_white_recall': mean_white_r,
    'mean_black_recall': mean_black_r,
    'seed_white_recall': seed_white_recall,
    'seed_black_recall': seed_black_recall,
    'failures': failures,
    'model_name': model_name,
}

with open(os.path.join(config.CLASS_OUT, 'classification_results_all_models.json'), 'w') as f:
    json.dump({'neural': neural_results}, f, indent=2)
print('\nLLaMA-3-8B done. Results saved.')


## 4. Explainability: SHAP · LIME · Integrated Gradients (DeBERTa / Mistral)

We'll use standard text explainers for our trained complex models. Note that for LLMs like Mistral-7B, explainability methods can be computationally intensive, so it's advised to explain a small subset.

In [3]:
# ─── Load Best DeBERTa Model for Explainability ───────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import json

explain_model_name = 'microsoft/deberta-v3-small'

# Find the best checkpoint dynamically from trainer_state.json
best_model_path = explain_model_name
base_dir = os.path.join(config.BERT_DIR, f'deberta-v3-small_seed42')
if os.path.exists(base_dir):
    # Find the most recent checkpoint directory to read its trainer_state.json
    chkpts = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if d.startswith('checkpoint-')]
    if chkpts:
        latest_chkpt = max(chkpts, key=os.path.getmtime)
        state_file = os.path.join(latest_chkpt, 'trainer_state.json')
        if os.path.exists(state_file):
            with open(state_file, 'r') as f:
                state = json.load(f)
                best_chkpt = state.get('best_model_checkpoint')
                if best_chkpt and os.path.exists(best_chkpt):
                    best_model_path = best_chkpt

print(f"Loading best model from: {best_model_path}")

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(explain_model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    best_model_path,
    num_labels=2, id2label={0:'White', 1:'Black'}, label2id={'White':0, 'Black':1}
).to(device)
model.eval()

text_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1, return_all_scores=True)

# Sample a unified test set for explainability
# Increase sample size to 50 per class for more robust explanation analysis
explain_sample = pd.concat([
    test_df[test_df['label'] == 1].head(50),
    test_df[test_df['label'] == 0].head(50)
])
explain_sample['doc_id'] = explain_sample.index

explain_texts = explain_sample['text'].tolist()
explain_labels = explain_sample['label'].tolist()
doc_ids = explain_sample['doc_id'].tolist()

# Store outputs here
explain_results = {
    str(did): {'text': txt, 'true_label': int(lbl), 'SHAP': {}, 'LIME': {}, 'IG': {}} 
    for did, txt, lbl in zip(doc_ids, explain_texts, explain_labels)
}
print(f"Prepared {len(explain_texts)} samples for explainability.")


Loading best model from: ../../models/bert/deberta-v3-small_seed42/checkpoint-284


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Prepared 100 samples for explainability.


In [4]:
# ─── 4A. Integrated Gradients via Captum ───────────────────────────────────────
from captum.attr import LayerIntegratedGradients
import numpy as np

def ig_forward(input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask).logits

# For DeBERTa, the word embeddings layer is typically model.deberta.embeddings.word_embeddings
try:
    lig = LayerIntegratedGradients(ig_forward, model.deberta.embeddings.word_embeddings)
except AttributeError:
    # fallback for other architectures
    lig = LayerIntegratedGradients(ig_forward, model.get_input_embeddings())

IG_STEPS = 50
MAX_LEN = 256

ig_records = []
from tqdm.notebook import tqdm

print("Running Integrated Gradients...")
for doc_id, text, label in tqdm(zip(doc_ids, explain_texts, explain_labels), total=len(doc_ids)):
    # Prepare inputs
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    
    baseline_ids = torch.zeros_like(input_ids)
    baseline_mask = torch.zeros_like(attention_mask)
    
    attributions, delta = lig.attribute(
        inputs=(input_ids, attention_mask),
        baselines=(baseline_ids, baseline_mask),
        n_steps=IG_STEPS,
        target=label,
        return_convergence_delta=True
    )
    
    token_attr = attributions[0].sum(dim=-1).squeeze(0).cpu().detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())
    
    valid = [(t.replace(' ', ''), float(a)) for t, a, m in zip(tokens, token_attr, attention_mask[0].cpu()) 
             if m == 1 and t not in [tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token]]
    
    # Store token scores in dictionary
    explain_results[str(doc_id)]['IG'] = valid
    
    top_pos = sorted(valid, key=lambda x: x[1], reverse=True)[:10]
    top_neg = sorted(valid, key=lambda x: x[1])[:10]
    ig_records.append({'doc_id': doc_id, 'true_label': label, 'pos': top_pos, 'neg': top_neg})

# Display a single sample's top words
print("Captum IG Top Positive Tokens for Black prediction (Sample 1):", ig_records[0]['pos'])

Running Integrated Gradients...


  0%|          | 0/100 [00:00<?, ?it/s]

Captum IG Top Positive Tokens for Black prediction (Sample 1): [('▁On', 0.12133488801675811), ('▁and', 0.0632766360823012), ('▁Ave', 0.060344975083406506), ('▁above', 0.048799882245538204), ('▁Warwick', 0.045935740473804214), ('▁trapped', 0.04535422811100432), ('▁people', 0.04272749058099372), ('▁accident', 0.041103381325332505), ('.', 0.03572742607263653), ('.', 0.027876718132142563)]


In [5]:
# ─── 4B. SHAP (SHapley Additive exPlanations) ──────────────────────────────────
import shap
import scipy as sp

# Using the pipeline for SHAP Text Explainer
shap_explainer = shap.Explainer(text_pipeline)

# Explain the sample (Running over all sampled texts)
print("Running SHAP (this may take a few moments)...")
shap_values = shap_explainer(explain_texts)

# Extract and store the SHAP values grouped by text
for i, doc_id in enumerate(doc_ids):
    # shap_values[i].values holds attributions. For text, it often gives values for each class, so shape is (tokens, classes)
    # We focus on the attribution for predicting label = 1 (Black). If binary, it might be 2D.
    # Check shape to adapt dynamically:
    sv = shap_values[i].values
    if len(sv.shape) == 2:
        sv_class1 = sv[:, 1]
    else:
        sv_class1 = sv
    
    tokens = shap_values[i].data
    
    # Pair tokens with their SHAP weights for the positive class (Black)
    token_weights = [(str(t).replace(' ', ''), float(v)) for t, v in zip(tokens, sv_class1) if str(t).strip() != '']
    explain_results[str(doc_id)]['SHAP'] = token_weights

# Visualize first text in sample
try:
    shap.plots.text(shap_values[0])
except Exception as e:
    print(f"SHAP Error: {e}")

Token indices sequence length is longer than the specified maximum sequence length for this model (783 > 512). Running this sequence through the model will result in indexing errors


Running SHAP (this may take a few moments)...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   1%|          | 1/100 [00:00<?, ?it/s]

...

PartitionExplainer explainer:  97%|█████████▋| 97/100 [13:00<00:13,  4.66s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 100/100 [13:18<00:00,  5.04s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 101it [13:27,  8.08s/it]                         


In [6]:
# ─── 4C. LIME (Local Interpretable Model-agnostic Explanations) ────────────────
from lime.lime_text import LimeTextExplainer
import torch.nn.functional as F

lima_explainer = LimeTextExplainer(class_names=['White', 'Black'])

def lime_predict_proba(texts):
    # LIME requires a function returning numpy arrays of probabilities
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probas = F.softmax(logits, dim=1).cpu().numpy()
    return probas

# Explain single sample
print("Running LIME...")
for i, (doc_id, text) in tqdm(enumerate(zip(doc_ids, explain_texts)), total=len(doc_ids)):
    # Limit number of features and samples for speed, though 100 samples is a bit small for extreme accuracy.
    lime_exp = lima_explainer.explain_instance(
        text, lime_predict_proba, num_features=25, num_samples=150
    )
    
    # Store word-weight pairs
    word_weights = [(w, float(weight)) for w, weight in lime_exp.as_list()]
    explain_results[str(doc_id)]['LIME'] = word_weights

# Show LIME results for a single sample
print("LIME Explanation for Sample 1:")
for word, weight in explain_results[str(doc_ids[0])]['LIME']:
    print(f"{word}: {weight:.4f}")

# ─── SAVE EXPLAINABILITY RESULTS ──────────────────────────────────────────────
output_path = os.path.join(config.CLASS_OUT, 'explainability_results.json')
with open(output_path, 'w') as f:
    json.dump(explain_results, f, indent=2)
print(f"\nAll explainability results saved to: {output_path}")

Running LIME...


  0%|          | 0/100 [00:00<?, ?it/s]

LIME Explanation for Sample 1:
was: -0.0562
intervene: 0.0448
people: 0.0430
him: 0.0323
handcuffing: 0.0300
disengaged: 0.0273
Sgt: 0.0255
same: 0.0220
for: -0.0208
had: -0.0206
patrol: 0.0204
vehicle: 0.0172
crossing: 0.0167
transported: 0.0165
and: -0.0165
took: 0.0160
TWC: -0.0153
while: 0.0139
scene: 0.0134
is: -0.0092
actively: -0.0077
away: 0.0070
completing: -0.0049
handcuffs: 0.0035
right: -0.0008

All explainability results saved to: ../../outputs/classification/explainability_results.json
